**READING From Bronze Layer and cleaning Transformed**

In [0]:
%python
drug_df = spark.read.table("clinical_trials_drug_data.bronze.drug_info_data") #drug_id
outcomes_df = spark.read.table("clinical_trials_drug_data.bronze.outcomes_data") #trial_id
patient_df = spark.read.table("clinical_trials_drug_data.bronze.patient_data") #patient_id
timeline_df = spark.read.table("clinical_trials_drug_data.bronze.timeline_data")  #trial_id
trial_df = spark.read.table("clinical_trials_drug_data.bronze.trial_data") #trial_id

In [0]:
drug_df3.show()

**DROPING DUPLICATES**

In [0]:
    %python
drug_df1 = drug_df.dropDuplicates()
outcomes_df1 = outcomes_df.dropDuplicates()
patient_df1 = patient_df.dropDuplicates()
timeline_df1 = timeline_df.dropDuplicates()
trial_df1 = trial_df.dropDuplicates()

In [0]:
trial_df1.display()

In [0]:
patient_df1.show()

**Filtering Data is not null**

In [0]:
from pyspark.sql.functions import col

drug_df2 = drug_df1.filter(col("drug_id").isNotNull())
outcomes_df2 = outcomes_df1.filter(col("trial_id").isNotNull())
outcomes_df2.createOrReplaceTempView("outcomes_df")
patient_df2 = patient_df1.filter(col("patient_id").isNotNull())
timeline_df2 = timeline_df1.filter(col("trial_id").isNotNull())
trial_df2 = trial_df1.filter(col("trial_id").isNotNull())
trial_df2.createOrReplaceTempView("trial_df")

**Changing data to upper case**

In [0]:
%python

from pyspark.sql.functions import upper, trim, lower

patient_df3 = patient_df2.withColumn("gender", upper(trim(col("gender"))))
patient_df3 = patient_df2.withColumn("disease", upper(trim(col("disease"))))

drug_df3 = drug_df2.withColumn("approval_authority", upper(col("approval_authority")))
drug_df3.createOrReplaceTempView("drug_df")

**changing data to date type**

In [0]:
%python
from pyspark.sql.functions import to_date

timeline_df3 = timeline_df2 \
    .withColumn("enrollment_date", to_date(col("enrollment_date"))) \
    .withColumn("treatment_start_date", to_date(col("treatment_start_date"))) \
    .withColumn("treatment_end_date", to_date(col("treatment_end_date")))   

**calculate date from star to end date for treatment**

In [0]:
from pyspark.sql.functions import datediff

timeline_df4 = timeline_df3.withColumn("treatment_duration", datediff(col("treatment_end_date"), col("treatment_start_date")))

In [0]:
timeline_df4.display()
timeline_df4.createOrReplaceTempView("timeline_df")

In [0]:
from pyspark.sql.functions import when

patient_df4 = patient_df3.withColumn("age_group", when(col("age") < 30 ,"Young") .when(col("age") <= 60 , "Adult") .otherwise("Senior"))
patient_df4.createOrReplaceTempView("patient_df")
patient_df4.display()


In [0]:
drug_df3.show()

**Join tables**

In [0]:
%python
silver_df = spark.sql("select trial_df.trial_id, trial_df.study_id, trial_df.study_phase, timeline_df.trial_id as outcome_trial, timeline_df.enrollment_date, timeline_df.treatment_start_date, timeline_df.treatment_end_date, timeline_df.treatment_duration, outcomes_df.trial_id as outcome_trial_id, outcomes_df.outcome, outcomes_df.adverse_effect, outcomes_df.cost_usd from trial_df join timeline_df on trial_df.trial_id = timeline_df.trial_id join outcomes_df on timeline_df.trial_id = outcomes_df.trial_id ")
silver_df.display()


**Writing tables back to silver layer**

In [0]:
silver_df.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.silver.silver_data")
drug_df3.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.silver.drug_silver_data")
patient_df4.write.mode("overwrite").saveAsTable("clinical_trials_drug_data.silver.patient_silver_data")

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("study_id") \
    .saveAsTable("clinical_trials_drug_data.silver.silver_data")


In [0]:
silver_df.display()

In [0]:
last_run_time = '2023-01-02'


In [0]:
incremental_df = silver_df.filter(
    col("enrollment_date") > last_run_time
)


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "clinical_trials_drug_data.silver.silver_data")

delta_table.alias("target").merge(
    incremental_df.alias("source"),
    "target.trial_id = source.trial_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [0]:
%sql
OPTIMIZE clinical_trials_drug_data.silver.silver_data;